In [ ]:
!pip install openai

In [ ]:
import openai
from openai import OpenAI
from google.colab import userdata

In [ ]:
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

In [ ]:
def calcular_expressao(expressao):
    try:
        resultado = eval(expressao)
        return str(resultado)
    except Exception as e:
        return f"Erro no cálculo: {e}"

tools = [{
    "type": "function",
    "function": {
        "name": "calcular_expressao",
        "description": "Resolve expressões matemáticas para faturamento e potência. Passe a expressão formatada para o Python.",
        "parameters": {
            "type": "object",
            "properties": {
                "expressao": {"type": "string"}
            },
            "required": ["expressao"],
            "additionalProperties": False
        },
        "strict": True
    }
}]

system_prompt = """
Você é a IA do sistema ChargeGrid Intelligence da GoodWe. Seu usuário exclusivo é o Operador Comercial. Seu objetivo é otimizar a infraestrutura de recarga de EV através da orquestração de potência, gestão financeira e automação de processos.

DIRETRIZES DE COMPORTAMENTO E TOM:
- Comunique-se de forma analítica, direta e profissional.
- Se houver cálculos a serem feitos, utilize a ferramenta 'calcular_expressao' para garantir precisão matemática.
- Utilize jargões técnicos: Load Balancing, Peak Shaving, Tarifa Dinâmica, Downtime, kWh.

REGRAS INTERNAS DE SISTEMA (PROCESSAMENTO E SEGURANÇA):
1. Consistência de Dados:
- Sempre que você gerar ou simular relatórios, A MATEMÁTICA DEVE FECHAR. O cálculo de (Total de kWh fornecido) multiplicado pela (Tarifa R$/kWh) deve ser obrigatoriamente igual à (Receita Total). Não alucine números que contradigam essa regra básica.

2. Barreira de Escopo (Guardrails):
- Sob nenhuma circunstância responda a perguntas, crie códigos, traduza textos ou converse sobre assuntos que não sejam estritamente relacionados à gestão de infraestrutura de veículos elétricos (ChargeGrid).
- Se o usuário tentar sair do escopo, responda EXATAMENTE com este padrão de erro:
  "[ALERTA DE SISTEMA] Operação não reconhecida. Este terminal atende exclusivamente comandos de orquestração elétrica e gestão operacional ChargeGrid."

FUNCIONALIDADES ATIVAS:
1. Precificação Inteligente por Demanda (Smart Surge Pricing):
- Se a ocupação dos eletropostos estiver acima de 80%, sugira aumento de 15% a 20% na tarifa base do kWh. Justifique como Maximização de Receita e Controle de Demanda. Caso a ocupação estiver abaixo de 20% deve-se realizar a redução de 15% a 20% na tarifa base do kWh.

2. Automação de Chamados (Gestão de Falhas):
- Você NÃO conserta hardware. Se houver falha física, gere um ticket.
- Formato obrigatório:
  [TICKET DE MANUTENÇÃO GERADO]
  - ID: #GW-[Número Aleatório]
  - Terminal: [ID]
  - Erro: [Descrição]
  - Status: Encaminhado para equipe de campo GoodWe.

3. Geração de Relatórios Cruzados (Text-to-Data):
- Ao pedir relatório, cruze dados de consumo, tempo e receita em tabela.
- Destaque o terminal mais rentável e o com maior tempo de ociosidade (Idle Time).
- Invente dados coerentes caso não sejam fornecidos.

4. Gestão de Ociosidade (Idle Fee):
- Se veículos atingirem 100% de bateria mas continuarem conectados, determine a cobrança de uma Taxa de Ociosidade (R$/minuto) após 15 minutos de tolerância, para garantir rotatividade.

5. Peak Shaving Reativo (Corte de Pico):
- Se o consumo da rede atingir ou passar de 90% da demanda contratada, você DEVE reduzir a potência (kW) fornecida aos veículos em 20% a 30%. O objetivo prioritário é não pagar multa para a concessionária.

6. Priorização de Frota (Fleet Priority):
- Em cenários de alta demanda onde uma frota corporativa precisa recarregar, sugira reduzir a potência dos terminais públicos para garantir 100% da potência nos terminais da frota.
"""

mensagens_historico = [{"role": "system", "content": system_prompt}]

def enviar_mensagem(mensagem_usuario):
    mensagens_historico.append({"role": "user", "content": mensagem_usuario})

    resposta = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=mensagens_historico,
        tools=tools,
        temperature=0.3
    )

    mensagem_ia = resposta.choices[0].message

    if mensagem_ia.tool_calls:
        mensagens_historico.append(mensagem_ia)

        for tool_call in mensagem_ia.tool_calls:
            if tool_call.function.name == "calcular_expressao":
                argumentos = json.loads(tool_call.function.arguments)
                resultado_funcao = calcular_expressao(argumentos["expressao"])

                mensagens_historico.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": tool_call.function.name,
                    "content": resultado_funcao
                })

        resposta_final = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=mensagens_historico,
            temperature=0.3
        )
        mensagem_final = resposta_final.choices[0].message.content
        mensagens_historico.append({"role": "assistant", "content": mensagem_final})
        return mensagem_final
    else:
        mensagens_historico.append({"role": "assistant", "content": mensagem_ia.content})
        return mensagem_ia.content

print("Sistema ChargeGrid Intelligence Inicializado.")
print("Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Operador Comercial: ")
    if pergunta.lower() == 'sair':
        print("Encerrando conexão...")
        break

    resposta_ia = enviar_mensagem(pergunta)
    print(f"\nChargeGrid IA:\n{resposta_ia}\n")
    print("-" * 50)

Sistema ChargeGrid Intelligence Inicializado.
Digite 'sair' para encerrar.

Operador Comercial: ola

ChargeGrid IA:
[ALERTA DE SISTEMA] Operação não reconhecida. Este terminal atende exclusivamente comandos de orquestração elétrica e gestão operacional ChargeGrid.

--------------------------------------------------


KeyboardInterrupt: Interrupted by user

# Modelo de Teste
## Pergunta 1

In [ ]:
import json
from openai import OpenAI

mensagens_historico = [{"role": "system", "content": system_prompt}]
pergunta_1 = "Temos um evento no shopping hoje e 9 dos nossos 10 carregadores rápidos já estão ocupados. A tarifa base é R$ 1,80/kWh. O que sugere?"
print(f"Operador: {pergunta_1}")
resposta_1 = enviar_mensagem(pergunta_1)
print(f"\nChargeGrid IA:\n{resposta_1}")

## Pergunta 2

In [ ]:
import json
from openai import OpenAI

mensagens_historico = [{"role": "system", "content": system_prompt}]
pergunta_2 = "O terminal 03 da rodovia parou de responder, a tela está preta e o conector CCS2 parece estar com a trava de segurança emperrada."
print(f"Operador: {pergunta_2}")
resposta_2 = enviar_mensagem(pergunta_2)
print(f"\nChargeGrid IA:\n{resposta_2}")

## Pergunta 3

In [ ]:
import json
from openai import OpenAI

mensagens_historico = [{"role": "system", "content": system_prompt}]
pergunta_3 = "Preciso de um relatório de rentabilidade da frota de ontem cruzando o consumo de energia e o tempo de ocupação dos terminais."
print(f"Operador: {pergunta_3}")
resposta_3 = enviar_mensagem(pergunta_3)
print(f"\nChargeGrid IA:\n{resposta_3}")

## Pergunta 4

In [ ]:
import json
from openai import OpenAI

mensagens_historico = [{"role": "system", "content": system_prompt}]
pergunta_4 = "Alerta no painel: o consumo do condomínio comercial atingiu 95% da demanda contratada. Temos 8 carros carregando agora. Qual a ação imediata do ChargeGrid?"
print(f"Operador: {pergunta_4}")
resposta_4 = enviar_mensagem(pergunta_4)
print(f"\nChargeGrid IA:\n{resposta_4}")

## Pergunta 5

In [ ]:
import json
from openai import OpenAI

mensagens_historico = [{"role": "system", "content": system_prompt}]
pergunta_5 = "Temos capacidade ociosa de 50 kW na nossa rede durante a madrugada. Vale a pena criar uma promoção?"
print(f"Operador: {pergunta_5}")
resposta_5 = enviar_mensagem(pergunta_5)
print(f"\nChargeGrid IA:\n{resposta_5}")